# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cooper30/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Decision:** Which content items should an editor review first for a possible refresh?

This notebook uses March 2026 information to rank the probability of a substantial April 2026 impressions decline. It is a decision-support prototype, not evidence that refreshing a page causes recovery.

## 1. Method choice and why

I use an interpretable `DecisionTreeClassifier` because the output must be explainable to an editor, not just accurate. I compare depth 2 and depth 3 trees and choose between them on a validation set. The final output is a ranked queue based on predicted probability, so ranking metrics are primary.

The target is one when April impressions are below 80% of March impressions. This is an observed directional decline proxy, not a causal refresh label. To reduce noise, the frame includes pages with at least 100 March impressions and clients with at least 20 available GSC days in both months.

The five model features are available at the March 31 decision point: log March impressions, March CTR, March average position, March observed days, and content age. April data is used only to construct the outcome.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy

import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

SEED = 42
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Add your Hugging Face READ token to Colab Secrets as HF_TOKEN.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

model_frame = con.execute(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_march,
        SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_position_march,
        COUNT(DISTINCT report_date) AS observed_days_march
    FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april
    FROM {FACT_APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
march_coverage AS (
    SELECT client_hash_id, COUNT(DISTINCT report_date) AS available_days_march
    FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id
),
april_coverage AS (
    SELECT client_hash_id, COUNT(DISTINCT report_date) AS available_days_april
    FROM {FACT_APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.impressions_march,
    m.clicks_march,
    m.ctr_march,
    m.avg_position_march,
    m.observed_days_march,
    DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
    COALESCE(a.impressions_april, 0) AS impressions_april
FROM march m
JOIN march_coverage mc USING (client_hash_id)
JOIN april_coverage ac USING (client_hash_id)
LEFT JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
LEFT JOIN {DIM_CONTENT} c
    ON m.content_hash_id = c.content_hash_id
WHERE mc.available_days_march >= 20
  AND ac.available_days_april >= 20
  AND m.impressions_march >= 100
  AND c.content_created_date IS NOT NULL
  AND c.content_created_date <= DATE '2026-03-31'
""").df()

model_frame["decline_ratio"] = model_frame["impressions_april"] / model_frame["impressions_march"]
model_frame["decline_label"] = (model_frame["decline_ratio"] < 0.80).astype(int)
model_frame["log_impressions_march"] = np.log1p(model_frame["impressions_march"])

FEATURES = [
    "log_impressions_march",
    "ctr_march",
    "avg_position_march",
    "observed_days_march",
    "content_age_days",
]

assert model_frame[FEATURES].notna().all().all()
assert model_frame["client_hash_id"].nunique() >= 5
print(f"Modeling rows: {len(model_frame):,}")
print(f"Clients: {model_frame['client_hash_id'].nunique():,}")
print(f"Observed decline base rate: {model_frame['decline_label'].mean():.3f}")


## 2. Split design

I split by pseudonymized client, not by page. No client appears in more than one split, which reduces the risk that client-specific patterns are memorized and then rewarded in evaluation. I use 60% of clients for training, 20% for validation, and 20% for the final test. Depth is selected only on validation; the test set is used once for the final comparison.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(SEED)
clients = np.array(sorted(model_frame["client_hash_id"].unique()))
rng.shuffle(clients)
n_test = max(1, int(round(len(clients) * 0.20)))
n_val = max(1, int(round(len(clients) * 0.20)))
test_clients = set(clients[:n_test])
val_clients = set(clients[n_test:n_test + n_val])
train_clients = set(clients[n_test + n_val:])

model_frame["split"] = np.where(
    model_frame["client_hash_id"].isin(test_clients), "test",
    np.where(model_frame["client_hash_id"].isin(val_clients), "validation", "train")
)
assert train_clients.isdisjoint(val_clients)
assert train_clients.isdisjoint(test_clients)
assert val_clients.isdisjoint(test_clients)

def ranking_metrics(y_true, score, ks=(20, 50)):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    order = np.argsort(-score, kind="mergesort")
    ranked_y = y_true[order]
    positives = max(int(y_true.sum()), 1)
    result = {
        "rows": int(len(y_true)),
        "base_rate": float(y_true.mean()),
        "average_precision": float(average_precision_score(y_true, score)),
        "roc_auc": float(roc_auc_score(y_true, score)),
    }
    for k in ks:
        kk = min(k, len(ranked_y))
        top = ranked_y[:kk]
        discounts = 1.0 / np.log2(np.arange(2, kk + 2))
        dcg = float((top * discounts).sum())
        ideal = np.sort(y_true)[::-1][:kk]
        idcg = float((ideal * discounts).sum())
        result[f"precision_at_{k}"] = float(top.mean())
        result[f"recall_at_{k}"] = float(top.sum() / positives)
        result[f"ndcg_at_{k}"] = float(dcg / idcg) if idcg else 0.0
    return result

display(model_frame.groupby("split").agg(rows=("decline_label", "size"), clients=("client_hash_id", "nunique"), base_rate=("decline_label", "mean")))


## 3. Train + compare with the Week-4 baseline

The Week-4 baseline gives two points for age ≥180 days, two for March impressions ≥500, and one-point bonuses for age ≥365 days and impressions ≥5,000. The baseline and model are evaluated on the exact same held-out client test rows and the same future-decline label.

Model selection uses validation `Precision@50`, with average precision as a tie-breaker. The selected tree is then refit on train + validation clients before the one-time test comparison.

In [ ]:
train_df = model_frame[model_frame["split"] == "train"].copy()
val_df = model_frame[model_frame["split"] == "validation"].copy()
test_df = model_frame[model_frame["split"] == "test"].copy()

validation_rows = []
for depth in (2, 3):
    candidate = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=200, random_state=SEED)
    candidate.fit(train_df[FEATURES], train_df["decline_label"])
    val_score = candidate.predict_proba(val_df[FEATURES])[:, 1]
    validation_rows.append({"max_depth": depth, **ranking_metrics(val_df["decline_label"], val_score)})

validation_table = pd.DataFrame(validation_rows).sort_values(["precision_at_50", "average_precision"], ascending=False)
display(validation_table)
selected_depth = int(validation_table.iloc[0]["max_depth"])

train_val_df = model_frame[model_frame["split"].isin(["train", "validation"])].copy()
final_model = DecisionTreeClassifier(max_depth=selected_depth, min_samples_leaf=200, random_state=SEED)
final_model.fit(train_val_df[FEATURES], train_val_df["decline_label"])
test_df["model_probability"] = final_model.predict_proba(test_df[FEATURES])[:, 1]

test_df["baseline_action_score"] = 0
test_df.loc[test_df["content_age_days"] >= 180, "baseline_action_score"] += 2
test_df.loc[test_df["impressions_march"] >= 500, "baseline_action_score"] += 2
test_df.loc[test_df["content_age_days"] >= 365, "baseline_action_score"] += 1
test_df.loc[test_df["impressions_march"] >= 5000, "baseline_action_score"] += 1
test_df["baseline_rank_score"] = test_df["baseline_action_score"] + 1e-3 * np.log1p(test_df["impressions_march"]) + 1e-7 * test_df["content_age_days"]
test_df["model_rank_score"] = test_df["model_probability"] + 1e-9 * test_df["impressions_march"].rank(pct=True)

baseline_metrics = ranking_metrics(test_df["decline_label"], test_df["baseline_rank_score"])
model_metrics = ranking_metrics(test_df["decline_label"], test_df["model_rank_score"])
comparison = pd.DataFrame([
    {"method": "Week-4 rule baseline", **baseline_metrics},
    {"method": f"Decision Tree depth {selected_depth}", **model_metrics},
])
display(comparison)

feature_importance = pd.DataFrame({"feature": FEATURES, "importance": final_model.feature_importances_}).sort_values("importance", ascending=False)
display(feature_importance)

metrics_receipt = {
    "lane": "Refresh / Content Opportunity Scoring",
    "observation_window": "2026-03",
    "outcome_window": "2026-04",
    "target": "April impressions below 80% of March impressions",
    "minimum_march_impressions": 100,
    "minimum_available_days_per_client_month": 20,
    "split": "client-grouped 60/20/20 train/validation/test",
    "random_seed": SEED,
    "features": FEATURES,
    "selected_depth": selected_depth,
    "validation_candidates": validation_table.to_dict(orient="records"),
    "test_baseline": baseline_metrics,
    "test_model": model_metrics,
    "feature_importance": feature_importance.to_dict(orient="records"),
    "leakage_checks": {"future_fields_used_as_features": False, "ids_used_as_features": False, "client_overlap_across_splits": False}
}

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_DIR / "w05_model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_receipt, f, indent=2)

ranked_test = test_df.sort_values("model_rank_score", ascending=False).copy()
ranked_test.insert(0, "model_rank", np.arange(1, len(ranked_test) + 1))
ranked_test.to_csv(OUTPUT_DIR / "w05_ranked_test_queue.csv", index=False)
print(f"Selected depth: {selected_depth}")
print("Metrics written to work/outputs/w05_model_metrics.json")


## 4. Errors and interpretation

False positives are pages ranked highly even though the April decline proxy is zero. They are not necessarily bad editorial suggestions: the proxy can miss pages that are stale or strategically important, but they count as errors for this measured task. False negatives are observed April declines ranked near the bottom; these reveal decline patterns the five-feature tree does not capture.

Feature importance is descriptive of this fitted tree and dataset. It does not establish that a feature causes search performance to change.

In [ ]:
error_view = ranked_test[["model_rank", "content_hash_id", "model_probability", "decline_label", "decline_ratio", "impressions_march", "ctr_march", "avg_position_march", "content_age_days"]].copy()
false_positives = error_view[error_view["decline_label"] == 0].head(10)
false_negatives = error_view[error_view["decline_label"] == 1].tail(10)
print("Highest-ranked false positives")
display(false_positives)
print("Lowest-ranked observed declines")
display(false_negatives)

for forbidden in ["impressions_april", "decline_ratio", "decline_label", "client_hash_id", "content_hash_id"]:
    assert forbidden not in FEATURES
assert Path("work/outputs/w05_model_metrics.json").exists()
print("SELF-CHECK PASSED ✅")
print("No future fields, labels, or IDs were used as model features.")

from google.colab import files
files.download("work/outputs/w05_model_metrics.json")


## Self-check

After **Runtime → Run all** completes, confirm:

- [ ] Every section above has visible output
- [ ] The notebook runs top to bottom with no errors
- [ ] The client sets do not overlap
- [ ] Baseline and model use the same held-out rows and target
- [ ] Base rate appears next to Precision@K
- [ ] No client names, domains, URLs, private queries, or credentials appear
- [ ] `w05_model_metrics.json` was downloaded and committed under `work/outputs/`
- [ ] Claims remain observed / measured / directional / decision-support